# Function 7

In [1]:

import numpy as np
import matplotlib.pyplot as plt
import math
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from scipy.stats import qmc



### Data preparation

Here we prepare the initial data provided

In [2]:
input = np.load('./initial_inputs.npy')
output = np.load('./initial_outputs.npy')
print(input)
print(output)
#print(input.shape)
#print(input.shape[1])
func_dimensions = input.shape[1]
print('this function has ', func_dimensions, ' dimensions')

[[0.27262382 0.32449536 0.89710881 0.83295115 0.15406269 0.79586362]
 [0.54300258 0.9246939  0.34156746 0.64648585 0.71844033 0.34313266]
 [0.09083225 0.66152938 0.06593091 0.25857701 0.96345285 0.6402654 ]
 [0.11886697 0.61505494 0.90581639 0.8553003  0.41363143 0.58523563]
 [0.63021764 0.8380969  0.68001305 0.73189509 0.52673671 0.34842921]
 [0.76491917 0.25588292 0.60908422 0.21807904 0.32294277 0.09579366]
 [0.05789554 0.49167222 0.24742222 0.21811844 0.42042833 0.73096984]
 [0.19525188 0.07922665 0.55458046 0.17056682 0.01494418 0.10703171]
 [0.64230298 0.83687455 0.02179269 0.10148801 0.68307083 0.6924164 ]
 [0.78994255 0.19554501 0.57562333 0.07365919 0.25904917 0.05109986]
 [0.52849733 0.45742436 0.36009569 0.36204551 0.81689098 0.63747637]
 [0.72261522 0.01181284 0.06364591 0.16517311 0.07924415 0.35995166]
 [0.07566492 0.33450212 0.13273274 0.60831236 0.91838592 0.82233079]
 [0.94245084 0.37743962 0.48612233 0.22879108 0.08263175 0.71195755]
 [0.14864702 0.03394336 0.72880565

Here we add the data provided with the weekly queries

In [3]:
additionalInputs = [[0.5, 0.5, 0.5, 0.5, 0.5, 0.5], [0.064108, 0.50467, 0.307547, 0.22933, 0.356495, 0.789273], [0.045203, 0.493887, 0.296157, 0.140072, 0.360602, 0.761422], [0.129753, 0.463468, 0.317766, 0.191937, 0.396281, 0.784369], [0.06782, 0.434119, 0.249574, 0.196558, 0.385974, 0.837422]]
additionalOutputs = [np.float64(0.5053149917022333), np.float64(1.5242238626243496), np.float64(1.348343558184188), np.float64(1.5755656391558395), np.float64(1.372546781524056)]

input = np.append(input, additionalInputs, axis=0)
output = np.append(output, additionalOutputs)

print(input)
print(output)

[[0.27262382 0.32449536 0.89710881 0.83295115 0.15406269 0.79586362]
 [0.54300258 0.9246939  0.34156746 0.64648585 0.71844033 0.34313266]
 [0.09083225 0.66152938 0.06593091 0.25857701 0.96345285 0.6402654 ]
 [0.11886697 0.61505494 0.90581639 0.8553003  0.41363143 0.58523563]
 [0.63021764 0.8380969  0.68001305 0.73189509 0.52673671 0.34842921]
 [0.76491917 0.25588292 0.60908422 0.21807904 0.32294277 0.09579366]
 [0.05789554 0.49167222 0.24742222 0.21811844 0.42042833 0.73096984]
 [0.19525188 0.07922665 0.55458046 0.17056682 0.01494418 0.10703171]
 [0.64230298 0.83687455 0.02179269 0.10148801 0.68307083 0.6924164 ]
 [0.78994255 0.19554501 0.57562333 0.07365919 0.25904917 0.05109986]
 [0.52849733 0.45742436 0.36009569 0.36204551 0.81689098 0.63747637]
 [0.72261522 0.01181284 0.06364591 0.16517311 0.07924415 0.35995166]
 [0.07566492 0.33450212 0.13273274 0.60831236 0.91838592 0.82233079]
 [0.94245084 0.37743962 0.48612233 0.22879108 0.08263175 0.71195755]
 [0.14864702 0.03394336 0.72880565

# Bayesian Optimisation approach
We approach the study of this function with the Bayesian Optimisation
using and adaptation of the UCB acquisition function from required assignment 12.1

# Preparation of the exploration space
Here we prepare the exploration space of the function<br>
The values are increments of 0.01 bounded between 0 and 1 (included) => 101 values for each dimension of the space<br>
dimension of the current function is stored in <b>func_dimensions</b>

In [4]:
#Initialise query lists and maximum observations
X, Y = input, output

# Latin Hypercube Sampling (LHS) ---
n_samples = 12500000
sampler_lhs = qmc.LatinHypercube(d=func_dimensions, seed=42)
x_grid = sampler_lhs.random(n=n_samples)
#print(f"LHS shape: {x_grid_lhs.shape}")

#x_grid = np.delete(x_grid, 0, axis=0)
#print(x_grid)
print(x_grid.shape)


(12500000, 6)


In [5]:
print(x_grid[:10])

[[0.91555962 0.06905196 0.79361561 0.8684149  0.00128071 0.64935552]
 [0.82320026 0.9158529  0.79337951 0.64520372 0.23297693 0.08541785]
 [0.99754547 0.67687697 0.83839308 0.01812326 0.5104818  0.51081127]
 [0.32755345 0.93484363 0.82821434 0.25922573 0.4843528  0.20291425]
 [0.80165738 0.7129927  0.45440348 0.88574528 0.94925767 0.10029611]
 [0.9087121  0.78399816 0.39667085 0.73034181 0.02814996 0.61689558]
 [0.56528799 0.5602366  0.2085167  0.91637971 0.56387837 0.86986657]
 [0.48673458 0.3851647  0.59128305 0.51255474 0.67121133 0.1763847 ]
 [0.17139459 0.35507295 0.2707955  0.33714024 0.73812882 0.40159027]
 [0.08941626 0.95959554 0.6377926  0.56167467 0.50765583 0.88114759]]


# Bayesian Optimisation with UCB applied

In [6]:
rbf_lengthscale = [0.1, 0.1, 0.1, 0.1, 0.1, 0.1]
real_noise_std = 1e-10
noise_assumption = 1e-10

#Define kernel of GP
kernel = RBF(length_scale=rbf_lengthscale, length_scale_bounds='fixed')

model = GaussianProcessRegressor(kernel = kernel)
#Fit the model
model.fit(np.array(X), np.array(Y).reshape(-1, 1))


#Calculate the mean and standard deviation and make them one-dimensional for plotting
post_mean, post_std = model.predict(x_grid, return_std=True)

#Acquisition function parameter
beta = 1.96
#beta = 0.5

acquisition_function = post_mean + beta * post_std

grid = x_grid.squeeze()
obs = grid[np.argmax(acquisition_function)] #Else use the acquisition function

print('Next observation: ', obs)
#print (obs)


Next observation:  [0.04263647 0.45963643 0.304379   0.22149697 0.43641056 0.79016618]
